# Freeze the NTC cell cycle score into a reusable reference (OPS0077)

This notebook turns the cell cycle phase score that was derived on the **NTC cells** into a **frozen reference** that can score *other* cells (for example gene knockouts) without using their own signals. It runs `scripts/ops0077_cc_transfer/build_ntc_reference.py`.

**What gets frozen (all estimated on NTC cells only)**
- per-well background, floor, median and MAD used to normalise the p21 / Rb / pRb nuclear intensities and the relative DNA content
- the closed cycle curve through `[phosphorylated Rb fraction, p21, pRb, DNA content]`, and the phase anchoring (0 = G1 start, increasing through S to G2/M)
- the **phase sectors** (G1, S, G2/M, M-to-G1 exit), defined from the curve's DNA content only
- a **phase-contrast model**: ridge circular regression from phase features to `(sin, cos)` of the phase

**Why it matters.** New cells are normalised with these NTC constants and projected onto the frozen curve. Renormalising a knockout population on itself would re-centre it and hide the shift being measured.

**Inputs** (relative to the repository root)

| file | content |
|---|---|
| `data/altair/ops0077_4i/OPS0077_NTC_OrganelleProfiler_pRb_Rb_p21.h5ad` | NTC 4i features |
| `data/altair/OPS0077_NTC_DAPI_cpmeasure.h5ad` | NTC DNA content |
| `data/altair/OPS0077_NTC_phase_OrganelleProfiler.h5ad` | NTC phase-contrast features |
| `reports/ops0077_cell_cycle_score/cell_cycle_scores.csv` | the existing NTC score, used only to **check** that the frozen reference reproduces it |

**Outputs** go to `reports/ops0077_cc_transfer/reference/`. Allow about 10 minutes (the phase model is cross-validated and refitted).

## 1. Requirements

In [ ]:
# %pip install anndata numpy pandas scipy scikit-learn joblib

## 2. Configuration and input checks

In [ ]:
import subprocess, sys, json
from pathlib import Path
ROOT = Path("..").resolve() if (Path("..") / "scripts").exists() else Path(".").resolve()
SCRIPTS = ROOT / "scripts/ops0077_cc_transfer"
def run(cmd):
    proc = subprocess.Popen([sys.executable, "-W", "ignore", *map(str, cmd)], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="")
    assert proc.wait() == 0, "failed, see the log above"

INPUTS = {
    "NTC 4i": ROOT / "data/altair/ops0077_4i/OPS0077_NTC_OrganelleProfiler_pRb_Rb_p21.h5ad",
    "NTC DNA content": ROOT / "data/altair/OPS0077_NTC_DAPI_cpmeasure.h5ad",
    "NTC phase contrast": ROOT / "data/altair/OPS0077_NTC_phase_OrganelleProfiler.h5ad",
    "existing NTC score (check only)": ROOT / "reports/ops0077_cell_cycle_score/cell_cycle_scores.csv",
}
OUT = ROOT / "reports/ops0077_cc_transfer/reference"
SKIP_PHASE_MODEL = False   # True: freeze only the stain reference (about 30 seconds)
for n, p in INPUTS.items():
    print(f"{'OK     ' if p.exists() else 'MISSING'} {n}: {p.relative_to(ROOT)}")
assert all(p.exists() for p in INPUTS.values()), "missing input(s)"

## 3. Build the reference

In [ ]:
cmd = [SCRIPTS / "build_ntc_reference.py", ROOT] + (["--skip-phase-model"] if SKIP_PHASE_MODEL else [])
run(cmd)

## 4. Checks

In [ ]:
import pandas as pd
summary = json.load(open(OUT / "reference_summary.json"))
print(json.dumps(summary, indent=2))
rep = summary["reproduction_vs_existing_cc_phase_dna"]
assert rep["max_abs_circ_diff"] < 1e-6, "the frozen reference does not reproduce the existing NTC score"
print("\nOK: the frozen reference reproduces the existing NTC phase for", rep["n_compared"], "cells")

`reproduction_vs_existing_cc_phase_dna` shows that re-deriving the score with the frozen-constant code gives the existing NTC `cc_phase_dna` to machine precision. `phase_model_tile_cv` is the honest accuracy of the phase-contrast model on NTC cells it never saw (cross-validated by tile): the mean circular error in turns of a full cycle, where a random guess gives 0.25.

## 5. The phase sectors, defined from the NTC curve only

In [ ]:
ref = json.load(open(OUT / "ntc_cc_reference.json"))
print(pd.DataFrame(ref["sector_intervals"], columns=["sector", "phase_from", "phase_to"]).to_string(index=False))
print("\nNTC fraction of cells per sector:", ref["ntc_sector_fractions"])
print("NTC distance-to-curve 95th / 99th percentile:", round(ref["ntc_curve_dist_q95"], 2), round(ref["ntc_curve_dist_q99"], 2))

In [ ]:
import numpy as np, matplotlib.pyplot as plt
prof = pd.read_csv(OUT / "ntc_profiles_by_phase.csv")
fig, ax = plt.subplots(1, 4, figsize=(18, 3.6))
for a, (f, t) in zip(ax, [("z_dna", "DNA content (z)"), ("z_phos", "phosphorylated Rb (z)"), ("z_p21", "p21 (z)"), ("z_pRb", "pRb (z)")]):
    a.fill_between(prof["phase_mid"], prof[f + "_q25"], prof[f + "_q75"], alpha=.3); a.plot(prof["phase_mid"], prof[f + "_q50"], lw=2)
    for s, x0, x1 in ref["sector_intervals"]:
        a.axvspan(x0, x1, alpha=.06, color={"G1": "C0", "S": "C1", "G2/M": "C3", "M-to-G1 exit": "C2"}[s])
    a.set_title(t); a.set_xlabel("NTC phase")
plt.tight_layout()

## 6. Output files

| file | content |
|---|---|
| `ntc_cc_reference.json` | frozen constants, curve nodes, anchoring, sectors (plain JSON, safe to share) |
| `ntc_reference_scores.csv` | NTC cells re-scored with the frozen reference |
| `ntc_profiles_by_phase.csv` | NTC marker and DNA profiles along the phase |
| `phase_ridge_model.joblib`, `phase_model_features.json` | phase-contrast model and its feature list. **A joblib file is a Python pickle: only load files you trust, or rebuild it with this notebook** |
| `reference_summary.json` | reproduction check and model cross-validation |

Next: `19_run_ops0077_cc_phase_transfer_to_new_cells.ipynb` applies this reference to new cells.